In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, recall_score, precision_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve,
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
df = pd.read_csv("Heart_Disease_Prediction.csv")
print(df.shape)
display(df.head())
df.info()
print("\nПропуски:\n", df.isna().sum())
print("\nНули:\n", (df.select_dtypes(include="number") == 0).sum())
print("\nЦель:\n", df["Heart Disease"].value_counts())
df.describe()


In [ ]:
RULES = {
    "BP": {"lo": 70, "hi": 250, "type": "cont", "src": "AHA/ACC 2025"},
    "Cholesterol": {"lo": 100, "hi": 600, "type": "cont", "src": "NHLBI ATP III"},
    "Max HR": {"lo": 60, "hi": 220, "type": "cont", "src": "ACC/AHA / JACC"},
    "ST depression": {"lo": 0, "hi": 10, "type": "cont", "src": "AHA exercise standards"},
    "Chest pain type": {"allowed": {1, 2, 3, 4}, "type": "cat", "src": "UCI"},
    "FBS over 120": {"allowed": {0, 1}, "type": "cat", "src": "dataset + ADA"},
    "EKG results": {"allowed": {0, 1, 2}, "type": "cat", "src": "UCI"},
    "Exercise angina": {"allowed": {0, 1}, "type": "cat", "src": "UCI"},
    "Slope of ST": {"allowed": {1, 2, 3}, "type": "cat", "src": "UCI"},
    "Number of vessels fluro": {"allowed": {0, 1, 2, 3}, "type": "cat", "src": "UCI"},
    "Thallium": {"allowed": {3, 6, 7}, "type": "cat", "src": "UCI"},
}

if "Cholesterol" in df.columns:
    df["Cholesterol"] = df["Cholesterol"].replace(0, np.nan)

for col, r in RULES.items():
    if col not in df.columns:
        continue
    if r["type"] == "cont":
        mask = df[col].notna() & ((df[col] < r["lo"]) | (df[col] > r["hi"]))
    else:
        mask = df[col].notna() & (~df[col].isin(r["allowed"]))
    print(f"{col}: вне диапазона {int(mask.sum())} | {r['src']}")
    if mask.any():
        display(df.loc[mask, [col]])
    df.loc[mask, col] = np.nan

print("\nNaN после очистки:\n", df.isna().sum())


In [ ]:
le = LabelEncoder()
df["Heart Disease"] = le.fit_transform(df["Heart Disease"].astype(str))
print("Классы:", dict(zip(le.classes_, le.transform(le.classes_))))

X = df.drop(columns=["Heart Disease"])
y = df["Heart Disease"]
print(X.shape, y.value_counts().to_dict())


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True).round(3))


In [ ]:
medians = X_train.median(numeric_only=True)
X_train = X_train.fillna(medians)
X_test = X_test.fillna(medians)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("NaN:", X_train.isna().sum().sum(), X_test.isna().sum().sum())


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "KNN": KNeighborsClassifier(),
    "SVM (RBF)": SVC(
        kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

NEED_SCALE = {"Logistic Regression", "KNN", "SVM (RBF)"}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

results = []
for name, model in models.items():
    X_cv = X_train_scaled if name in NEED_SCALE else X_train.values
    scores = cross_validate(model, X_cv, y_train, cv=cv, scoring=scoring)
    results.append({
        "Модель": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1": scores["test_f1"].mean(),
        "ROC-AUC": scores["test_roc_auc"].mean(),
    })

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
display(results_df.round(3))
print("Лучшая по CV ROC-AUC:", results_df.iloc[0]["Модель"])


In [ ]:
gs = GridSearchCV(
    LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    {"C": [0.01, 0.1, 1, 10]},
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
)
gs.fit(X_train_scaled, y_train)
print("Лучшие параметры:", gs.best_params_)
print("CV ROC-AUC:", round(gs.best_score_, 3))

best_model = gs.best_estimator_
X_fit, X_eval = X_train_scaled, X_test_scaled


In [ ]:
def pick_threshold(y_true, proba, min_precision=0.65):
    prec, rec, thr = precision_recall_curve(y_true, proba)
    prec, rec, thr = prec[:-1], rec[:-1], thr
    cands = [(r, p, t) for p, r, t in zip(prec, rec, thr) if p >= min_precision]
    if cands:
        r, p, t = max(cands, key=lambda x: (x[0], x[1]))
        return float(t), float(r), float(p)
    f2 = (5 * prec * rec) / (4 * prec + rec + 1e-12)
    i = int(np.argmax(f2))
    return float(thr[i]), float(rec[i]), float(prec[i])


MIN_PRECISION = 0.65  # фиксировано по вашему выбору

y_train_proba = best_model.predict_proba(X_fit)[:, 1]
y_test_proba = best_model.predict_proba(X_eval)[:, 1]

best_thr, train_rec, train_prec = pick_threshold(
    y_train, y_train_proba, min_precision=MIN_PRECISION
)

print(f"MIN_PRECISION = {MIN_PRECISION}")
print(f"Выбранный порог: {best_thr:.3f}  (вместо 0.5)")
print(f"На train → Presence recall={train_rec:.3f}, precision={train_prec:.3f}")

pred_05 = (y_test_proba >= 0.5).astype(int)
pred_new = (y_test_proba >= best_thr).astype(int)

cmp = pd.DataFrame([
    {
        "вариант": "порог 0.5",
        "threshold": 0.5,
        "Presence_recall": round(recall_score(y_test, pred_05, pos_label=1), 3),
        "Presence_precision": round(precision_score(y_test, pred_05, pos_label=1), 3),
        "Accuracy": round(accuracy_score(y_test, pred_05), 3),
        "FN_пропуски_больных": int(confusion_matrix(y_test, pred_05)[1, 0]),
        "FP_ложные_тревоги": int(confusion_matrix(y_test, pred_05)[0, 1]),
    },
    {
        "вариант": f"порог {best_thr:.3f} (MIN_PRECISION=0.65)",
        "threshold": round(best_thr, 3),
        "Presence_recall": round(recall_score(y_test, pred_new, pos_label=1), 3),
        "Presence_precision": round(precision_score(y_test, pred_new, pos_label=1), 3),
        "Accuracy": round(accuracy_score(y_test, pred_new), 3),
        "FN_пропуски_больных": int(confusion_matrix(y_test, pred_new)[1, 0]),
        "FP_ложные_тревоги": int(confusion_matrix(y_test, pred_new)[0, 1]),
    },
])
display(cmp)


In [ ]:
def show(y_true, y_hat, title):
    print(f"\n=== {title} ===")
    print("Presence recall   :", round(recall_score(y_true, y_hat, pos_label=1), 3))
    print("Presence precision:", round(precision_score(y_true, y_hat, pos_label=1), 3))
    print("Accuracy          :", round(accuracy_score(y_true, y_hat), 3))
    print("F1 Presence       :", round(f1_score(y_true, y_hat, pos_label=1), 3))
    print("ROC-AUC           :", round(roc_auc_score(y_true, y_test_proba), 3))
    print(classification_report(y_true, y_hat, target_names=le.classes_))

show(y_test, pred_05, "Порог 0.5")
show(y_test, pred_new, f"Порог {best_thr:.3f} (MIN_PRECISION=0.65)")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_05), display_labels=le.classes_).plot(
    ax=axes[0], cmap="Blues", colorbar=False
)
axes[0].set_title("Порог 0.5")
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_new), display_labels=le.classes_).plot(
    ax=axes[1], cmap="Greens", colorbar=False
)
axes[1].set_title(f"Порог {best_thr:.3f} (mp=0.65)")
plt.tight_layout()
plt.show()

y_pred = pred_new


In [ ]:
train_auc = roc_auc_score(y_train, best_model.predict_proba(X_fit)[:, 1])
test_auc = roc_auc_score(y_test, y_test_proba)
print(f"Train AUC: {train_auc:.3f}")
print(f"Test  AUC: {test_auc:.3f}")
print(f"Gap:       {train_auc - test_auc:.3f}")


Если нужно вообще 0 пропусков больных — можно временно поставить `MIN_PRECISION = 0.55`.
